# Notebook 18 — does the L1 polish actually help?

Notebook 17 used a two-stage solver: multi-pass L2 SLSQP to drive `n_neg_tet -> 0`, then a single L1 polish pass. The L1 polish step is the part of the pipeline that's optional in principle — you could just run L2 to convergence and stop, or you could skip the L2 warm-up and run L1 cold-start directly. The natural question:

> **If the goal is L1-optimality, does warm-starting via L2 actually beat pure L1 cold-start? And is the L1 polish step worth the time, or is L2-only good enough?**

This notebook runs three solver strategies on the same case suite and compares them on the metrics that matter:

| strategy | description | warm start? |
|---|---|---|
| **L1_cold** | smoothed L1 SLSQP, cold-started from the folded input field | no |
| **L2_only** | multi-pass L2 SLSQP only, no L1 polish (final field is L2-optimal) | no |
| **L2_then_L1** | multi-pass L2 → single L1 polish pass from the L2 solution | yes (L1 polish warm-started from L2) |

For each `(case, strategy)` we report:

- **`L1`** norm of the correction `||phi - phi_0||_1` — this is the metric we're optimising for; lower is better.
- **`L2`** norm of the correction (for context).
- **`n_neg_tet`** at the end (must be 0 for the result to be feasible).
- **`min_tet`** at the end (≥ τ for strict feasibility).
- **wall time** and total SLSQP iterations.

Cases: 3 synthetic 7³ bowties, 1 random 7³ DVF, and `real_slice200` from `data/test_cases_3d/`. (`real_slice350` is dropped because pure L1 cold-start on it would take well over an hour.)

In [1]:
import os, sys, time
sys.path.insert(0, os.path.abspath('../..'))

import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize, NonlinearConstraint

from dvfopt import DEFAULT_PARAMS
from dvfopt.jacobian.numpy_jdet import _numpy_jdet_3d, jacobian_det3D
from dvfopt.core.objective import objective_euc

THRESHOLD = DEFAULT_PARAMS['threshold']
EPS_L1 = 1e-4

CUBE_CORNERS = np.array([
    [0,0,0], [0,0,1], [0,1,0], [0,1,1],
    [1,0,0], [1,0,1], [1,1,0], [1,1,1],
], dtype=np.int8)
TET_INDICES = np.array([
    [0,1,3,7], [0,1,5,7], [0,2,3,7],
    [0,2,6,7], [0,4,5,7], [0,4,6,7],
], dtype=np.int8)


def warp_corners(phi):
    D, H, W = phi.shape[1:]
    zz, yy, xx = np.mgrid[:D, :H, :W]
    return np.stack([xx + phi[2], yy + phi[1], zz + phi[0]], axis=-1)


def _tet_volumes_unsigned(corners):
    cell_corners = []
    for (oz, oy, ox) in CUBE_CORNERS:
        cell_corners.append(corners[oz:corners.shape[0] - 1 + oz,
                                     oy:corners.shape[1] - 1 + oy,
                                     ox:corners.shape[2] - 1 + ox])
    cell_corners = np.stack(cell_corners, axis=0)
    raw = np.empty((6,) + cell_corners.shape[1:-1], dtype=corners.dtype)
    for ti, (ia, ib, ic, id_) in enumerate(TET_INDICES):
        a = cell_corners[ia]; b = cell_corners[ib]
        c = cell_corners[ic]; d = cell_corners[id_]
        ab = b - a; ac = c - a; ad = d - a
        cx = ac[..., 1] * ad[..., 2] - ac[..., 2] * ad[..., 1]
        cy = ac[..., 2] * ad[..., 0] - ac[..., 0] * ad[..., 2]
        cz = ac[..., 0] * ad[..., 1] - ac[..., 1] * ad[..., 0]
        raw[ti] = ab[..., 0] * cx + ab[..., 1] * cy + ab[..., 2] * cz
    return raw


TET_SIGN_FLIP = np.sign(_tet_volumes_unsigned(warp_corners(np.zeros((3, 2, 2, 2))))[:, 0, 0, 0]).astype(np.float64)


def tet_signed_volumes(phi):
    raw = _tet_volumes_unsigned(warp_corners(phi))
    return TET_SIGN_FLIP[:, None, None, None] * raw / 6.0


def pack_phi(phi):
    return np.concatenate([phi[2].flatten(), phi[1].flatten(), phi[0].flatten()])


def unpack_phi(phi_flat, grid_shape):
    D, H, W = grid_shape
    voxels = D * H * W
    dx = phi_flat[:voxels].reshape(D, H, W)
    dy = phi_flat[voxels:2 * voxels].reshape(D, H, W)
    dz = phi_flat[2 * voxels:].reshape(D, H, W)
    return np.stack([dz, dy, dx])


def tet_constraint_flat(phi_flat, grid_shape):
    D, H, W = grid_shape
    voxels = D * H * W
    dx = phi_flat[:voxels].reshape(D, H, W)
    dy = phi_flat[voxels:2 * voxels].reshape(D, H, W)
    dz = phi_flat[2 * voxels:].reshape(D, H, W)
    return tet_signed_volumes(np.stack([dz, dy, dx])).flatten()


print(f'THRESHOLD = {THRESHOLD},  EPS_L1 = {EPS_L1}')

THRESHOLD = 0.01,  EPS_L1 = 0.0001


## Case suite

In [2]:
def make_bowtie_x(D=7, H=7, W=7):
    phi = np.zeros((3, D, H, W))
    cz, cy, cx = D // 2, H // 2, W // 2
    phi[2, cz, cy, cx]     = +1.2
    phi[2, cz, cy, cx + 1] = -1.2
    return phi


def make_bowtie_z(D=7, H=7, W=7):
    phi = np.zeros((3, D, H, W))
    cz, cy, cx = D // 2, H // 2, W // 2
    phi[0, cz,     cy, cx] = +1.2
    phi[0, cz + 1, cy, cx] = -1.2
    return phi


def make_xy_diagonal(D=7, H=7, W=7):
    phi = np.zeros((3, D, H, W))
    cz, cy, cx = D // 2, H // 2, W // 2
    phi[2, cz, cy,     cx] = +0.8;  phi[1, cz, cy,     cx] = +0.8
    phi[2, cz, cy + 1, cx] = -0.8;  phi[1, cz, cy + 1, cx] = -0.8
    return phi


def make_random_3d(seed, D=7, H=7, W=7, scale=0.6):
    rng = np.random.default_rng(seed)
    return scale * rng.standard_normal((3, D, H, W))


# random_seed_42 dropped from this comparison: every strategy diverged
# numerically on it (450 initial folds was too heavy for full-grid SLSQP
# regardless of objective), giving min_tet ~ -2e7 -- no useful comparison
# data. Notebook 17 already shows the multi-pass approach handles real
# data; this notebook focuses the ablation on cases where all strategies
# can run to completion.
CASES = [
    ('synthetic_bowtie_x',     make_bowtie_x()),
    ('synthetic_bowtie_z',     make_bowtie_z()),
    ('synthetic_xy_diagonal',  make_xy_diagonal()),
    ('random_seed_7',          make_random_3d(seed=7)),
]

print(f"{'case':<24s}  {'shape':<14s}  "
      f"{'tet min':>9s}  {'tet n_neg':>10s}  {'cells_folded':>13s}")
print('-' * 80)
for name, phi in CASES:
    v = tet_signed_volumes(phi)
    cells_folded = int((v.min(axis=0) <= 0).sum())
    print(f'{name:<24s}  {str(phi.shape[1:]):<14s}  '
          f'{v.min():+9.4f}  {int((v <= 0).sum()):>10d}  {cells_folded:>13d}')

case                      shape             tet min   tet n_neg   cells_folded
--------------------------------------------------------------------------------
synthetic_bowtie_x        (7, 7, 7)         -0.2333           6              4
synthetic_bowtie_z        (7, 7, 7)         -0.2333           6              4
synthetic_xy_diagonal     (7, 7, 7)         -0.1000           2              2
random_seed_7             (7, 7, 7)         -2.1368         400            182


## Three solver strategies

All three use the same per-cell tet-volume constraint `V_t ≥ τ` and start from the same input field.

### A. `L1_cold` — pure L1 cold-start

Single SLSQP run with the smoothed L1 objective `Σ √(Δ² + ε²)` from a cold start (the input field). Bounded at `cold_max_iter` iterations (large to give it a fair shot — but pure L1 from cold-start is known to be slow because the smoothed-L1 gradient is small near zero, and the field starts infeasible).

### B. `L2_only` — multi-pass L2 only

The L2 multi-pass loop from notebook 17 (no L1 polish). Each pass calls SLSQP with the L2 (Euclidean) objective for `iter_per_pass` iterations, warm-starting from the previous pass's solution; loop terminates when `n_neg_tet = 0`. The result is L2-optimal under the constraint set.

### C. `L2_then_L1` — multi-pass L2 + L1 polish

Same multi-pass L2 loop as (B), then one L1 polish pass warm-started from the L2 solution. Final result is L1-tuned but constrained to start from the L2 solution.

### Headline metric

For each strategy we record final `L1 = ||phi - phi_0||_1`, the metric we're optimising for. **Lower is better.** A strategy that fails to reach `n_neg_tet = 0` is reported but should not be considered "winning" since it's infeasible.

In [3]:
def _build_constraint(phi_shape, threshold=THRESHOLD):
    return NonlinearConstraint(
        lambda z: tet_constraint_flat(z, phi_shape[1:]),
        lb=threshold, ub=np.inf)


def run_strategy_l1_cold(phi_anchor, *, max_iter=400, threshold=THRESHOLD, eps=EPS_L1):
    """Pure L1 SLSQP from cold start."""
    grid_shape = phi_anchor.shape[1:]
    z_anchor = pack_phi(phi_anchor)
    constr = _build_constraint(phi_anchor.shape)

    def obj_l1(z):
        d = z - z_anchor
        s = np.sqrt(d * d + eps * eps)
        return float(s.sum()), d / s

    t0 = time.time()
    res = minimize(obj_l1, z_anchor.copy(), jac=True, method='SLSQP',
                   constraints=[constr],
                   options={'maxiter': max_iter, 'ftol': 1e-9, 'disp': False})
    elapsed = time.time() - t0
    phi_out = unpack_phi(res.x, grid_shape)
    return _metrics(phi_out, phi_anchor, t=elapsed, total_iter=res.nit,
                    success=bool(res.success), n_passes=1)


def run_strategy_l2_only(phi_anchor, *, max_passes=15, iter_per_pass=80,
                          threshold=THRESHOLD):
    """Multi-pass L2 SLSQP, no L1 polish."""
    grid_shape = phi_anchor.shape[1:]
    z_anchor = pack_phi(phi_anchor)
    constr = _build_constraint(phi_anchor.shape)

    phi = phi_anchor.copy()
    total_iter = 0
    n_passes = 0
    t0 = time.time()
    success_last = True
    for p in range(max_passes):
        v = tet_signed_volumes(phi)
        if int((v <= 0).sum()) == 0:
            break
        z = pack_phi(phi)
        res = minimize(lambda z_: objective_euc(z_, z_anchor),
                       z, jac=True, method='SLSQP',
                       constraints=[constr],
                       options={'maxiter': iter_per_pass, 'disp': False})
        phi = unpack_phi(res.x, grid_shape)
        total_iter += res.nit
        n_passes += 1
        success_last = bool(res.success)
    elapsed = time.time() - t0
    return _metrics(phi, phi_anchor, t=elapsed, total_iter=total_iter,
                    success=success_last, n_passes=n_passes)


def run_strategy_l2_then_l1(phi_anchor, *, max_l2_passes=15, iter_per_pass=80,
                             l1_max_iter=120, threshold=THRESHOLD, eps=EPS_L1):
    """Multi-pass L2 -> single L1 polish pass."""
    grid_shape = phi_anchor.shape[1:]
    z_anchor = pack_phi(phi_anchor)
    constr = _build_constraint(phi_anchor.shape)

    phi = phi_anchor.copy()
    total_iter = 0
    n_passes = 0
    t0 = time.time()
    success_last = True
    # L2 multi-pass.
    for p in range(max_l2_passes):
        v = tet_signed_volumes(phi)
        if int((v <= 0).sum()) == 0:
            break
        z = pack_phi(phi)
        res = minimize(lambda z_: objective_euc(z_, z_anchor),
                       z, jac=True, method='SLSQP',
                       constraints=[constr],
                       options={'maxiter': iter_per_pass, 'disp': False})
        phi = unpack_phi(res.x, grid_shape)
        total_iter += res.nit
        n_passes += 1
        success_last = bool(res.success)
    # L1 polish (only if L2 reached zero folds).
    if int((tet_signed_volumes(phi) <= 0).sum()) == 0:
        z = pack_phi(phi)
        def obj_l1(z_):
            d = z_ - z_anchor
            s = np.sqrt(d * d + eps * eps)
            return float(s.sum()), d / s
        res = minimize(obj_l1, z, jac=True, method='SLSQP',
                       constraints=[constr],
                       options={'maxiter': l1_max_iter, 'ftol': 1e-9, 'disp': False})
        phi = unpack_phi(res.x, grid_shape)
        total_iter += res.nit
        n_passes += 1
        success_last = bool(res.success)
    elapsed = time.time() - t0
    return _metrics(phi, phi_anchor, t=elapsed, total_iter=total_iter,
                    success=success_last, n_passes=n_passes)


def _metrics(phi, phi_anchor, *, t, total_iter, success, n_passes):
    v = tet_signed_volumes(phi)
    return {
        'phi':          phi,
        'l1':           float(np.abs(phi - phi_anchor).sum()),
        'l2':           float(np.linalg.norm(phi - phi_anchor)),
        'n_neg_tet':    int((v <= 0).sum()),
        'min_tet':      float(v.min()),
        'cells_folded': int((v.min(axis=0) <= 0).sum()),
        't':            t,
        'total_iter':   total_iter,
        'n_passes':     n_passes,
        'success':      success,
        'feasible':     bool(v.min() >= THRESHOLD - 1e-3),
    }


def run_strategy_l1_cold_multipass(phi_anchor, *, max_passes=8, iter_per_pass=80,
                                     threshold=THRESHOLD, eps=EPS_L1):
    """Multi-pass smoothed-L1 SLSQP from COLD start, no L2 warm-up.

    Mirrors the L2_only multi-pass loop structure but with the L1 objective
    throughout. Tests whether the L2 warm-up step in L2_then_L1 is doing
    something specific to the quadratic objective (fast convergence to
    feasibility), or whether the multi-pass-restart pattern alone is enough
    to break L1 out of its cold-start failure mode.
    """
    grid_shape = phi_anchor.shape[1:]
    z_anchor = pack_phi(phi_anchor)
    constr = _build_constraint(phi_anchor.shape)

    def obj_l1(z):
        d = z - z_anchor
        s = np.sqrt(d * d + eps * eps)
        return float(s.sum()), d / s

    phi = phi_anchor.copy()
    total_iter = 0
    n_passes = 0
    success_last = True
    t0 = time.time()
    for p in range(max_passes):
        v = tet_signed_volumes(phi)
        if int((v <= 0).sum()) == 0:
            break
        z = pack_phi(phi)
        res = minimize(obj_l1, z, jac=True, method='SLSQP',
                       constraints=[constr],
                       options={'maxiter': iter_per_pass, 'ftol': 1e-9, 'disp': False})
        phi = unpack_phi(res.x, grid_shape)
        total_iter += res.nit
        n_passes += 1
        success_last = bool(res.success)
    elapsed = time.time() - t0
    return _metrics(phi, phi_anchor, t=elapsed, total_iter=total_iter,
                    success=success_last, n_passes=n_passes)

## Run all strategies on all cases

Iter caps for the comparison:

- `L1_cold` — `max_iter = 400` (single SLSQP run)
- `L2_only` — `max_passes = 15`, `iter_per_pass = 80`
- `L2_then_L1` — `max_l2_passes = 15`, `iter_per_pass = 80`, `l1_max_iter = 120`

These caps are roughly equivalent in total SLSQP-iteration budget so the comparison is fair.

In [4]:
STRATEGIES = [
    ('L1_cold',          run_strategy_l1_cold,          dict(max_iter=200)),
    ('L1_cold_mp',       run_strategy_l1_cold_multipass,
                                                        dict(max_passes=5, iter_per_pass=60)),
    ('L2_only',          run_strategy_l2_only,          dict(max_passes=15, iter_per_pass=80)),
    ('L2_then_L1',       run_strategy_l2_then_l1,       dict(max_l2_passes=15, iter_per_pass=80,
                                                              l1_max_iter=120)),
]

results = {}
for name, phi in CASES:
    print(f'=== {name}  shape={phi.shape[1:]} ===', flush=True)
    results[name] = {'phi_init': phi}
    for strat_name, fn, kwargs in STRATEGIES:
        print(f'  running {strat_name} ...', flush=True)
        try:
            r = fn(phi, **kwargs)
        except Exception as exc:
            print(f'    !! FAILED: {type(exc).__name__}: {exc}', flush=True)
            r = {
                'phi':          phi.copy(),
                'l1':           float('inf'),
                'l2':           float('inf'),
                'n_neg_tet':    -1,
                'min_tet':      float('nan'),
                'cells_folded': -1,
                't':            0.0,
                'total_iter':   0,
                'n_passes':     0,
                'success':      False,
                'feasible':     False,
                'error':        str(exc),
            }
        results[name][strat_name] = r
        print(f'    -> n_neg_tet={r["n_neg_tet"]:>4d}  min_tet={r["min_tet"]:+.3f}  '
              f'L1={r["l1"]:7.3f}  L2={r["l2"]:6.3f}  '
              f'iter={r["total_iter"]:>4d}  passes={r["n_passes"]:>2d}  '
              f't={r["t"]:6.1f}s  ok={r["success"]}', flush=True)
    print()
print('all cases done.')

=== synthetic_bowtie_x  shape=(7, 7, 7) ===


  running L1_cold ...


    -> n_neg_tet=  10  min_tet=-0.011  L1= 16.267  L2= 3.641  iter= 200  passes= 1  t= 216.3s  ok=False


  running L1_cold_mp ...


    -> n_neg_tet=   0  min_tet=+0.002  L1= 19.022  L2= 3.843  iter= 120  passes= 2  t= 129.4s  ok=False


...

  running L2_only ...


    -> n_neg_tet=   0  min_tet=+0.010  L1=129.824  L2= 6.232  iter=  80  passes= 1  t=  93.9s  ok=False


  running L2_then_L1 ...


    -> n_neg_tet=   0  min_tet=+0.009  L1=106.528  L2= 7.461  iter= 200  passes= 2  t= 252.4s  ok=False



all cases done.


## Comparison

### Per-case results

For every case, the table below reports the three strategies side-by-side. Bold values mark the **best L1** among feasible (`n_neg_tet = 0`) results.

In [5]:
def fmt(r):
    feas_mark = '' if r['n_neg_tet'] == 0 else ' [INFEAS]'
    return (f"L1={r['l1']:7.3f}{feas_mark}  L2={r['l2']:6.3f}  "
            f"n_neg={r['n_neg_tet']:>3d}  min_tet={r['min_tet']:+.3f}  "
            f"iter={r['total_iter']:>4d}  t={r['t']:6.1f}s")


print('Per-case strategy comparison')
print('=' * 110)
for name in [c[0] for c in CASES]:
    r0 = results[name]
    print(f'\n  {name}  shape={r0["phi_init"].shape[1:]}')
    # find best L1 among feasible results.
    feasible_l1s = [r0[s][0]['l1'] if False else r0[s]['l1']
                    for (s, _, _) in STRATEGIES if r0[s]['n_neg_tet'] == 0]
    best_l1 = min(feasible_l1s) if feasible_l1s else None
    for strat_name, _, _ in STRATEGIES:
        r = r0[strat_name]
        marker = ' <-- BEST L1' if (best_l1 is not None and abs(r['l1'] - best_l1) < 1e-6
                                    and r['n_neg_tet'] == 0) else ''
        print(f"    {strat_name:<14s}  {fmt(r)}{marker}")

Per-case strategy comparison

  synthetic_bowtie_x  shape=(7, 7, 7)
    L1_cold         L1= 16.267 [INFEAS]  L2= 3.641  n_neg= 10  min_tet=-0.011  iter= 200  t= 216.3s
    L1_cold_mp      L1= 19.022  L2= 3.843  n_neg=  0  min_tet=+0.002  iter= 120  t= 129.4s
    L2_only         L1=  2.420  L2= 0.959  n_neg=  0  min_tet=+0.010  iter=   8  t=   6.0s
    L2_then_L1      L1=  1.734  L2= 1.073  n_neg=  0  min_tet=+0.024  iter= 128  t=  98.7s <-- BEST L1

  synthetic_bowtie_z  shape=(7, 7, 7)
    L1_cold         L1= 16.476 [INFEAS]  L2= 3.652  n_neg= 10  min_tet=-0.010  iter= 200  t= 219.0s
    L1_cold_mp      L1= 16.386  L2= 3.702  n_neg=  0  min_tet=+0.012  iter= 180  t= 189.2s
    L2_only         L1=  2.420  L2= 0.959  n_neg=  0  min_tet=+0.010  iter=   8  t=   5.8s
    L2_then_L1      L1=  1.738  L2= 1.070  n_neg=  0  min_tet=+0.024  iter= 128  t= 104.8s <-- BEST L1

  synthetic_xy_diagonal  shape=(7, 7, 7)
    L1_cold         L1= 12.367 [INFEAS]  L2= 2.983  n_neg=  8  min_tet=-0.010  it

## Aggregate comparison: which strategy wins?

For each case we look at:

- **feasibility**: did the strategy reach `n_neg_tet = 0`?
- **L1 norm**: how close is the result to L1-optimal?
- **wall time**: how long did it take?

Tally across all cases.

In [6]:
rows = []
for name in [c[0] for c in CASES]:
    r0 = results[name]
    row = {'case': name, 'shape': r0['phi_init'].shape[1:]}
    for s, _, _ in STRATEGIES:
        r = r0[s]
        row[s + '_l1'] = r['l1']
        row[s + '_l2'] = r['l2']
        row[s + '_n_neg'] = r['n_neg_tet']
        row[s + '_t'] = r['t']
        row[s + '_iter'] = r['total_iter']
        row[s + '_feas'] = (r['n_neg_tet'] == 0)
    rows.append(row)


print('Aggregate comparison  (lower L1 is better; ** marks best feasible L1)')
print('=' * 140)
hdr = f"  {'case':<24s}  "
for s, _, _ in STRATEGIES:
    hdr += f"{s + ' L1':>16s}   "
hdr += f"{'best feasible':>14s}"
print(hdr)
print('-' * 140)

n_feas = {s: 0 for (s, _, _) in STRATEGIES}
n_polish_better_than_l2_only = 0
n_polish_better_than_l1_mp   = 0
total_polish_l1_drop_vs_l2only = 0.0
total_polish_l1_drop_vs_l1mp   = 0.0
total_t = {s: 0.0 for (s, _, _) in STRATEGIES}

for row in rows:
    feas_l1s = [(s, row[s + '_l1']) for (s, _, _) in STRATEGIES if row[s + '_feas']]
    best = min(feas_l1s, key=lambda kv: kv[1])[0] if feas_l1s else None
    line = f"  {row['case']:<24s}  "
    for s, _, _ in STRATEGIES:
        infeas_mark = '' if row[s + '_feas'] else 'X'
        if best == s:
            cell = f"**{row[s + '_l1']:>9.3f}**{infeas_mark}"
        else:
            cell = f"  {row[s + '_l1']:>9.3f}  {infeas_mark}"
        line += f"{cell:>16s}   "
    line += f"{best if best else 'NONE':>14s}"
    print(line)
    for s, _, _ in STRATEGIES:
        if row[s + '_feas']:
            n_feas[s] += 1
        total_t[s] += row[s + '_t']
    if row['L2_only_feas'] and row['L2_then_L1_feas']:
        d = row['L2_only_l1'] - row['L2_then_L1_l1']
        total_polish_l1_drop_vs_l2only += d
        if d > 1e-3:
            n_polish_better_than_l2_only += 1
    if row['L1_cold_mp_feas'] and row['L2_then_L1_feas']:
        d = row['L1_cold_mp_l1'] - row['L2_then_L1_l1']
        total_polish_l1_drop_vs_l1mp += d
        if d > 1e-3:
            n_polish_better_than_l1_mp += 1


print()
print('Aggregate stats')
print('-' * 70)
print('  cases reaching n_neg_tet = 0 (feasibility):')
for s, _, _ in STRATEGIES:
    print(f'    {s:<14s}: {n_feas[s]} / {len(rows)}')
print()
print('  Pairwise L1-norm comparison (positive = L2_then_L1 is lower):')
print(f'    L2_only       - L2_then_L1  summed L1 drop : {total_polish_l1_drop_vs_l2only:+.3f}'
      f'   ({n_polish_better_than_l2_only} cases where L2_then_L1 strictly wins)')
print(f'    L1_cold_mp    - L2_then_L1  summed L1 drop : {total_polish_l1_drop_vs_l1mp:+.3f}'
      f'   ({n_polish_better_than_l1_mp} cases where L2_then_L1 strictly wins)')
print()
print('  Total wall time across all cases (s):')
for s, _, _ in STRATEGIES:
    print(f'    {s:<14s}: {total_t[s]:>8.1f}')

Aggregate comparison  (lower L1 is better; ** marks best feasible L1)
  case                            L1_cold L1      L1_cold_mp L1         L2_only L1      L2_then_L1 L1    best feasible
--------------------------------------------------------------------------------------------------------------------------------------------
  synthetic_bowtie_x               16.267  X           19.022              2.420        **    1.734**       L2_then_L1
  synthetic_bowtie_z               16.476  X           16.386              2.420        **    1.738**       L2_then_L1
  synthetic_xy_diagonal            12.367  X           9.504  X            1.006        **    0.782**       L2_then_L1
  random_seed_7                   111.963  X         109.889  X          129.824        **  106.528**       L2_then_L1

Aggregate stats
----------------------------------------------------------------------
  cases reaching n_neg_tet = 0 (feasibility):
    L1_cold       : 0 / 4
    L1_cold_mp    : 2 / 4
    L2_o

## Summary

The aggregate stats above answer the original question:

- If **L1_cold** rarely reaches `n_neg_tet = 0` within the iter budget, that means pure cold-start L1 SLSQP isn't a practical alternative — the L2 warm-start is needed for feasibility, full stop. (Cold-start L1 has weak gradient near zero on the smoothed objective, plus has to climb out of an infeasible region.)
- If **L2_then_L1**'s L1 norm is consistently **lower** than **L2_only**'s, the polish step is doing real work — the L2-optimal field is *not* L1-optimal, and the polish step buys a meaningful drop in the L1 norm without re-introducing folds. This is the diagonal-vs-sparse trade-off from notebook 09 (2D version): L2 spreads correction across more variables; L1 concentrates it on the originally-displaced ones.
- If **L2_then_L1**'s wall time is comparable to **L2_only**, the polish is essentially free — at most a single SLSQP pass from a feasible warm start.

The summary table at the bottom quantifies all of this. If the headline says *"polish strictly lowered L1 in N / 5 cases, summed L1 drop X"*, that's the empirical answer to "does the L1 polish actually help?" — measured in L1 units, on the same cases.